In [ ]:
#@title Explorador de BIGGAN
#@markdown Celda 1 - Instalación y Verificación del entorno
# ============================================================
# BIGGAN EXPLORER
# Celda 1 - Instalación y verificación del entorno
# ============================================================

# Instalar BigGAN y dependencias
!pip -q install pytorch-pretrained-biggan

import torch
import numpy as np
import matplotlib.pyplot as plt

from pytorch_pretrained_biggan import (
    BigGAN,
    one_hot_from_names,
    truncated_noise_sample,
)

print("=" * 50)
print("BIGGAN EXPLORER")
print("=" * 50)

# Verificar GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Dispositivo: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memoria: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
else:
    print("⚠️ No se detectó GPU. Se recomienda ejecutar en Google Colab con GPU.")

print(f"PyTorch: {torch.__version__}")

In [ ]:
# ============================================================
#@markdown Celda 2 - Cargar BigGAN
# ============================================================

print("Cargando BigGAN...")

model = BigGAN.from_pretrained('biggan-deep-256')

model.to(device)
model.eval()
import requests

# Retrieve the list of 1000 ImageNet classes directly
url = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
try:
    response = requests.get(url)
    clases = response.json()
    print(f"✓ Loaded {len(clases)} ImageNet classes successfully!")
    print("Sample classes:", clases[:5])
except Exception as e:
    print("Error loading classes, using fallback list.")
    # Fallback to empty list or basic structure if offline
    clases = [f"Class {i}" for i in range(1000)]

print("✓ Modelo cargado correctamente")

In [3]:
# ============================================================
# @markdown Celda 3 - Función para generar imágenes con BigGAN
# ============================================================

import torch
import numpy as np
import matplotlib.pyplot as plt
from pytorch_pretrained_biggan import truncated_noise_sample

def generar(clase=0, seed=1, truncation=0.4):

    # Semilla
    torch.manual_seed(seed)

    # Vector latente
    noise = truncated_noise_sample(
        truncation=truncation,
        batch_size=1
    )

    noise = torch.tensor(noise).to(device)

    # Vector one-hot de 1000 clases
    label = np.zeros((1,1000), dtype=np.float32)
    label[0,clase] = 1

    label = torch.tensor(label).to(device)

    # Generar imagen
    with torch.no_grad():
        img = model(noise, label, truncation)

    # Mostrar
    img = img[0].cpu().permute(1,2,0)
    img = ((img+1)/2).clamp(0,1)

    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.axis("off")
    plt.show()
    plt.show()

In [ ]:
#@markdown Celda 4 - Encontrar Clases

# ============================================================
# BIGGAN LATENT SPACE EXPLORER
# Explorar una clase variando seed y truncation
# ============================================================

import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt
import numpy as np
import torch


def explorar_latente(
    palabra="jellyfish",
    seed=1,
    truncation=0.4
):

    # Buscar clase
    indices = [
        i for i,nombre in enumerate(clases)
        if palabra.lower() in nombre.lower()
    ]

    if len(indices)==0:
        print("No se encontró la clase")
        return

    categoria = indices[0]


    # Crear vector latente
    torch.manual_seed(seed)

    noise = truncated_noise_sample(
        truncation=truncation,
        batch_size=1
    )

    noise = torch.tensor(noise).to(device)


    # Vector clase
    label = np.zeros(
        (1,1000),
        dtype=np.float32
    )

    label[0,categoria] = 1

    label = torch.tensor(label).to(device)


    # Generar
    with torch.no_grad():
        img = model(
            noise,
            label,
            truncation
        )


    # Preparar imagen
    img = img[0].cpu().permute(1,2,0)
    img = ((img+1)/2).clamp(0,1)


    plt.figure(figsize=(6,6))
    plt.imshow(img)
    plt.axis("off")
    plt.title(
        f"{clases[categoria]}\nseed={seed} | trunc={truncation}"
    )
    plt.show()



# ------------------------------------------------
# Panel interactivo
# ------------------------------------------------

widgets.interact(
    explorar_latente,

    palabra=widgets.Text(
        value="jellyfish",
        description="Clase:"
    ),

    seed=widgets.IntSlider(
        value=1,
        min=0,
        max=1000,
        step=1,
        description="Seed:"
    ),

    truncation=widgets.FloatSlider(
        value=0.4,
        min=0.1,
        max=1,
        step=0.05,
        description="Trunc:"
    )
);


In [ ]:
#@markdown Celda 4.1 - Encontrar Clases

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import numpy as np
import torch


# -----------------------------
# Parámetros de usuario
# -----------------------------

clase = widgets.Text(
    value="jellyfish",
    description="Clase:"
)

inicio = widgets.IntSlider(
    value=1,
    min=0,
    max=1000,
    description="Seed inicial:"
)

cantidad = widgets.IntSlider(
    value=16,
    min=4,
    max=36,
    step=4,
    description="Cantidad:"
)

truncation = widgets.FloatSlider(
    value=0.4,
    min=0.1,
    max=1,
    step=0.1,
    description="Trunc:"
)


boton = widgets.Button(
    description="Explorar",
    button_style="primary"
)

salida = widgets.Output()


# -----------------------------
# Función de exploración
# -----------------------------

def explorar(b):

    with salida:
        clear_output(wait=True)

        # Buscar clase
        indices = [
            i for i,nombre in enumerate(clases)
            if clase.value.lower() in nombre.lower()
        ]

        if len(indices)==0:
            print("No se encontró la categoría")
            return

        categoria = indices[0]

        print(
            f"Clase: {categoria} → {clases[categoria]}"
        )


        n = cantidad.value

        columnas = int(np.sqrt(n))
        filas = int(np.ceil(n / columnas))

        fig, axes = plt.subplots(
            filas,
            columnas,
            figsize=(12,12)
        )

        axes = np.array(axes).reshape(-1)


        for i in range(n):

            seed = inicio.value + i

            torch.manual_seed(seed)

            noise = truncated_noise_sample(
                truncation=truncation.value,
                batch_size=1
            )

            noise = torch.tensor(noise).to(device)


            # Vector clase
            label = np.zeros(
                (1,1000),
                dtype=np.float32
            )

            label[0,categoria] = 1

            label = torch.tensor(label).to(device)


            # Generar
            with torch.no_grad():
                img = model(
                    noise,
                    label,
                    truncation.value
                )


            img = img[0].cpu().permute(1,2,0)
            img = ((img+1)/2).clamp(0,1)


            axes[i].imshow(img)
            axes[i].set_title(
                f"seed {seed}",
                fontsize=8
            )
            axes[i].axis("off")


        for ax in axes[n:]:
            ax.axis("off")


        plt.tight_layout()
        plt.show()



boton.on_click(explorar)


display(
    clase,
    inicio,
    cantidad,
    truncation,
    boton,
    salida
)

In [6]:
# ============================================================
#@markdown Celda 5 - BIGGAN - Preparar Mezlca de 3 clases
# ============================================================

def generar_blend_return(
    clase1,
    clase2,
    clase3,
    peso1=0.5,
    peso2=0.3,
    peso3=0.2,
    seed=1,
    truncation=0.4
):

    # Buscar clases
    ids = []

    for palabra in [clase1, clase2, clase3]:

        encontrados = [
            i for i,nombre in enumerate(clases)
            if palabra.lower() in nombre.lower()
        ]

        if len(encontrados) == 0:
            print("No encontrada:", palabra)
            return None

        ids.append(encontrados[0])


    # Crear vector semántico mezclado
    label = np.zeros((1,1000), dtype=np.float32)

    label[0,ids[0]] = peso1
    label[0,ids[1]] = peso2
    label[0,ids[2]] = peso3

    label = torch.tensor(label).to(device)


    # Crear latente
    torch.manual_seed(seed)

    noise = truncated_noise_sample(
        truncation=truncation,
        batch_size=1
    )

    noise = torch.tensor(noise).to(device)


    # Generar
    with torch.no_grad():

        img = model(
            noise,
            label,
            truncation
        )


    # Convertir a imagen
    img = img[0].cpu().permute(1,2,0)
    img = ((img+1)/2).clamp(0,1)


    return img

In [ ]:
# ============================================================
#@markdown Celda 6 - Visualizacion de 3 clases
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt


# -----------------------------
# Parámetros de clases
# -----------------------------

clase1 = widgets.Text(
    value="mushroom",
    description="Clase 1:"
)

clase2 = widgets.Text(
    value="coral reef",
    description="Clase 2:"
)

clase3 = widgets.Text(
    value="castle",
    description="Clase 3:"
)


# -----------------------------
# Pesos semánticos
# -----------------------------

peso1 = widgets.FloatSlider(
    value=0.5,
    min=0,
    max=1,
    step=0.05,
    description="Peso 1:"
)

peso2 = widgets.FloatSlider(
    value=0.3,
    min=0,
    max=1,
    step=0.05,
    description="Peso 2:"
)

peso3 = widgets.FloatSlider(
    value=0.2,
    min=0,
    max=1,
    step=0.05,
    description="Peso 3:"
)


# -----------------------------
# Espacio latente
# -----------------------------

seed = widgets.IntSlider(
    value=10,
    min=0,
    max=1000,
    description="Seed:"
)


truncation = widgets.FloatSlider(
    value=0.4,
    min=0.1,
    max=1,
    step=0.05,
    description="Trunc:"
)


# -----------------------------
# Tamaño visualización
# -----------------------------

tamano = widgets.IntSlider(
    value=512,
    min=256,
    max=1024,
    step=128,
    description="Tamaño:"
)


salida = widgets.Output()


# -----------------------------
# Actualizar imagen
# -----------------------------

def actualizar(change=None):

    with salida:

        clear_output(wait=True)

        img = generar_blend_return(
            clase1.value,
            clase2.value,
            clase3.value,
            peso1.value,
            peso2.value,
            peso3.value,
            seed.value,
            truncation.value
        )

        if img is not None:

            plt.figure(
                figsize=(
                    tamano.value/100,
                    tamano.value/100
                )
            )

            plt.imshow(img)

            plt.axis("off")

            plt.show()



# Actualizar cuando cambia cualquier parámetro

for control in [
    clase1,
    clase2,
    clase3,
    peso1,
    peso2,
    peso3,
    seed,
    truncation,
    tamano
]:

    control.observe(
        actualizar,
        names="value"
    )


display(
    clase1,
    peso1,
    clase2,
    peso2,
    clase3,
    peso3,
    seed,
    truncation,
    tamano,
    salida
)


# Primera generación
actualizar()

In [ ]:
# ============================================================
#@markdown Celda 7 - Explorador en Video del Espacio latente de BIGGAN
# Interpolación entre seeds + tamaño de salida
# ============================================================

import ipywidgets as widgets
from IPython.display import display, clear_output, Video

import numpy as np
import torch
import imageio

from PIL import Image



# ------------------------------------------------------------
# Generar frame desde latente
# ------------------------------------------------------------

def generar_frame(z, clase_id, truncation, tamaño):

    label = np.zeros(
        (1,1000),
        dtype=np.float32
    )

    label[0, clase_id] = 1

    label = torch.tensor(label).to(device)


    with torch.no_grad():

        img = model(
            z,
            label,
            truncation
        )


    img = img[0].cpu().permute(1,2,0)

    img = ((img+1)/2).clamp(0,1)


    # convertir a imagen
    img = (
        img.numpy()*255
    ).astype(np.uint8)


    img = Image.fromarray(img)


    # Escalar salida
    img = img.resize(
        (tamaño,tamaño),
        Image.LANCZOS
    )


    return np.array(img)



# ------------------------------------------------------------
# Panel de usuario
# ------------------------------------------------------------

clase_video = widgets.Text(
    value="jellyfish",
    description="Clase:"
)


num_seeds = widgets.IntSlider(
    value=5,
    min=2,
    max=10,
    step=1,
    description="Seeds:"
)


frames_seed = widgets.IntSlider(
    value=25,
    min=5,
    max=60,
    step=5,
    description="Frames:"
)


trunc_video = widgets.FloatSlider(
    value=0.4,
    min=0.1,
    max=1,
    step=0.05,
    description="Trunc:"
)


tamaño_video = widgets.Dropdown(
    options=[
        256,
        512,
        720,
        1080
    ],
    value=512,
    description="Salida:"
)


boton_video = widgets.Button(
    description="Crear video",
    button_style="primary"
)


salida_video = widgets.Output()



# ------------------------------------------------------------
# Crear video
# ------------------------------------------------------------

def crear_video(b):

    with salida_video:

        clear_output(wait=True)


        # Buscar clase

        encontrados = [
            i for i,nombre in enumerate(clases)
            if clase_video.value.lower()
            in nombre.lower()
        ]


        if len(encontrados)==0:

            print(
                "No se encontró la clase"
            )

            return


        clase_id = encontrados[0]


        print(
            "Clase:",
            clases[clase_id]
        )


        print(
            "Resolución:",
            tamaño_video.value,
            "px"
        )


        # Crear seeds

        seeds = np.random.randint(
            0,
            10000,
            num_seeds.value
        )


        print(
            "Seeds:",
            seeds
        )


        # Crear latentes

        latentes=[]


        for s in seeds:

            torch.manual_seed(
                int(s)
            )

            z = truncated_noise_sample(
                truncation=trunc_video.value,
                batch_size=1
            )


            z=torch.tensor(z).to(device)

            latentes.append(z)



        frames=[]


        # Interpolación

        for i in range(len(latentes)-1):

            z1 = latentes[i]
            z2 = latentes[i+1]


            for t in np.linspace(
                0,
                1,
                frames_seed.value
            ):


                z = (
                    (1-t)*z1 +
                    t*z2
                )


                frame = generar_frame(
                    z,
                    clase_id,
                    trunc_video.value,
                    tamaño_video.value
                )


                frames.append(frame)



        archivo="biggan_latent_exploration.mp4"


        imageio.mimsave(
            archivo,
            frames,
            fps=25
        )


        print(
            "Video creado:",
            archivo
        )


        display(
            Video(
                archivo,
                embed=True
            )
        )



boton_video.on_click(
    crear_video
)


display(
    clase_video,
    num_seeds,
    frames_seed,
    trunc_video,
    tamaño_video,
    boton_video,
    salida_video
)